# Speech Audit and Finalization — v1

Independent audit of the frozen v3 speech preprocessing outputs, re-verified from the saved files.
Ends by writing the **master recording-level QC table** (EEG + speech, 50 rows).
**Feature extraction and model training are out of scope and were not started.**

**Rules in force (non-negotiable):**
- Raw data (`P01`…`P10`) is read-only. Nothing is written, renamed or deleted there.
- Every new output is written **only** inside `E:\REU_DATA\audit_finalization_v1\` (guarded by `audit_write`).
- v3 outputs under `SPEECH processed\` are read-only for this pass.
- Quality statuses are recomputed with the **same metric thresholds applied uniformly to all 10 participants** — no participant-ID branches.
- Missing metadata is recorded as a gap, never invented.
- Every code cell is followed by a markdown cell with What / Why / Result bullets.

In [1]:
# STEP B0 - setup: imports, paths, safety guards, shared constants
from pathlib import Path
from math import gcd
import json, gc, hashlib, time, warnings
import numpy as np
import pandas as pd
import scipy.signal as signal
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RAW_ROOT     = Path(r"E:\REU_DATA")
AUDIT_DIR    = RAW_ROOT / "audit_finalization_v1"
SPEECH_OUT   = AUDIT_DIR / "speech"
REPORTS_OUT  = AUDIT_DIR / "reports"
FIG_OUT      = AUDIT_DIR / "figures"
SPEECH_PROC  = RAW_ROOT / "SPEECH processed" / "speech"
V3_REPORTS   = SPEECH_PROC / "reports"
BASELINE     = RAW_ROOT / "INVENTORY_BASELINE.csv"
EEG_AUDIT_QC = AUDIT_DIR / "eeg" / "eeg_audit_quality_control_v1.csv"
for d in (SPEECH_OUT, REPORTS_OUT, FIG_OUT):
    d.mkdir(parents=True, exist_ok=True)

def audit_write(path) -> Path:
    p = Path(path).resolve()
    root = AUDIT_DIR.resolve()
    if root != p and root not in p.parents:
        raise RuntimeError(f"refusing to write outside the audit directory: {p}")
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

PARTICIPANTS = [f"P{i:02d}" for i in range(1, 11)]
TARGET_SR    = 16000
CLIP_T       = 0.999
FRAME_S, HOP_S = 0.025, 0.010
NOISE_MARGIN_DB = 6.0
SNR_REVIEW_DB   = 15.0
LEVELDIFF_REVIEW_DB = 6.0

def speech_dir(pid):
    return RAW_ROOT / pid / f"{int(pid[1:]):02d}_speech"

def rms_db(v):
    return 20 * np.log10(np.sqrt((v ** 2).mean()) + 1e-20)

def frame_rms_db(x, sr):
    frame = max(int(FRAME_S * sr), 1); hop = max(int(HOP_S * sr), 1)
    n_f = max((len(x) - frame) // hop + 1, 1)
    out = np.empty(n_f)
    for i in range(n_f):
        seg = x[i * hop:i * hop + frame]
        out[i] = 20 * np.log10(np.sqrt((seg ** 2).mean()) + 1e-20)
    return out

def quality_metrics(x, sr):
    """Uniform metric battery applied identically to every recording."""
    mono = x.mean(axis=1) if x.ndim > 1 else x
    peak = float(np.abs(x).max())
    clipping_pct = float(np.mean(np.abs(x) >= CLIP_T) * 100)
    dc = float(mono.mean())
    db = frame_rms_db(mono, sr)
    floor = float(np.percentile(db, 10))
    speech_frames = db[db > floor + NOISE_MARGIN_DB]
    level = float(np.median(speech_frames)) if len(speech_frames) else float(np.percentile(db, 90))
    snr = level - floor
    return {"sample_rate_hz": int(sr), "n_channels": int(x.shape[1]) if x.ndim > 1 else 1,
            "duration_s": round(len(mono) / sr, 3), "peak": round(peak, 5),
            "peak_dbfs": round(20 * np.log10(peak + 1e-20), 2),
            "clipping_pct": round(clipping_pct, 5), "dc_offset": round(dc, 6),
            "noise_floor_db": round(floor, 2), "speech_level_db": round(level, 2),
            "estimated_snr_db": round(snr, 2),
            "low_snr_flag": bool(snr < SNR_REVIEW_DB),
            "clipping_flag": bool(clipping_pct > 0),
            "dc_flag": bool(abs(dc) > 0.01)}

if not EEG_AUDIT_QC.exists():
    raise RuntimeError("EEG audit QC table not found - run EEG_Audit_and_Finalization_v1.ipynb first")
print(f"RESULT: audit output root = {AUDIT_DIR}")
print(f"RESULT: EEG audit dependency found: {EEG_AUDIT_QC}")
t_start = time.time()

RESULT: audit output root = E:\REU_DATA\audit_finalization_v1
RESULT: EEG audit dependency found: E:\REU_DATA\audit_finalization_v1\eeg\eeg_audit_quality_control_v1.csv


### What was done
- Imported the libraries, defined the audit output root and the `audit_write` guard (no write outside `audit_finalization_v1`).
- Defined one uniform metric battery (`quality_metrics`) with fixed thresholds: clipping at 0.999 full scale, SNR review below 15 dB, DC review above 0.01, stereo level-difference review at 6 dB.
- Checked that the EEG audit table already exists (this notebook finishes the master QC table).

### Why
- The v3 speech review mixed metric-driven flags with participant-ID branches (`if pid == "P05"`). The audit recomputes every flag from data with the same rule for all 10 participants.
- The master table needs both audits, so the dependency is checked before any work starts.

### Result
- **Result**: the two printed paths confirm the output root and the EEG dependency. A missing EEG table stops this notebook immediately.

In [2]:
# STEP B1 - raw integrity pre-check for the 20 raw speech files vs INVENTORY_BASELINE.csv
base = pd.read_csv(BASELINE)
mask = base["RelPath"].str.contains(r"^P\d{2}\\\d{2}_speech\\", regex=True)
raw_files = base[mask].copy()
print(f"RESULT: raw speech files listed in the baseline: {len(raw_files)}")
rows = []
for _, r in raw_files.iterrows():
    p = RAW_ROOT / r["RelPath"]
    exists = p.exists()
    sha = ""
    if exists:
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 22), b""):
                h.update(chunk)
        sha = h.hexdigest()
    ok = bool(exists and sha == r["SHA256"])
    rows.append({"RelPath": r["RelPath"], "result": "PASS" if ok else "FAIL"})
    if not ok:
        print(f"RESULT: INTEGRITY FAILURE: {r['RelPath']}")
chk = pd.DataFrame(rows)
chk.to_csv(audit_write(REPORTS_OUT / "raw_integrity_speech_pre_v1.csv"), index=False)
n_fail = int((chk["result"] == "FAIL").sum())
print(f"RESULT: raw speech integrity pre-check: {len(chk)-n_fail}/{len(chk)} PASS, {n_fail} FAIL")
if n_fail or len(chk) != 20:
    raise RuntimeError("raw speech integrity check failed or unexpected file count - audit stopped")

RESULT: raw speech files listed in the baseline: 20


RESULT: raw speech integrity pre-check: 20/20 PASS, 0 FAIL


### What was done
- Re-hashed the 20 raw speech files (10 WAV + 10 AAC/MP4) against `INVENTORY_BASELINE.csv`.
- Stopped the notebook if any file differs or the count is not 20.

### Why
- Speech conclusions are only valid if the audio the audit compares against is byte-identical to the baseline.

### Result
- **Result**: expect `20/20 PASS`. Any FAIL aborts the notebook.

In [3]:
# STEP B2 - source inventory: WAV headers read from metadata; AAC/MP4 paired but never decoded
rows = []
for pid in PARTICIPANTS:
    d = speech_dir(pid)
    wavs = sorted(d.glob("*_wav.wav"))
    other = [p for p in d.iterdir() if p.suffix.lower() in (".aac", ".mp4", ".m4a")]
    rec = {"participant_id": pid, "n_wav": len(wavs), "n_companion": len(other),
           "companion_ext": other[0].suffix.lower() if other else ""}
    if wavs:
        info = sf.info(str(wavs[0]))
        rec.update({"source_wav": str(wavs[0]), "source_sr_hz": int(info.samplerate),
                    "source_channels": int(info.channels), "source_subtype": info.subtype,
                    "source_duration_s": round(info.frames / info.samplerate, 3),
                    "source_frames": int(info.frames)})
    rows.append(rec)
src_inv = pd.DataFrame(rows)
src_inv.to_csv(audit_write(SPEECH_OUT / "source_inventory_v1.csv"), index=False)
print(f"RESULT: source WAVs inventoried: {int((src_inv['n_wav']==1).sum())}/10 participants (exactly one each)")
print(f"RESULT: source sample rates: {src_inv.groupby('source_sr_hz')['participant_id'].apply(list).to_dict()}")
print(f"RESULT: source channel counts: mono {int((src_inv['source_channels']==1).sum())}, stereo {int((src_inv['source_channels']==2).sum())}")
print(f"RESULT: companion video/audio files present for {int((src_inv['n_companion']>0).sum())}/10 (duration pairing only; never decoded)")

RESULT: source WAVs inventoried: 10/10 participants (exactly one each)
RESULT: source sample rates: {44100: ['P01', 'P02', 'P03', 'P04', 'P05'], 48000: ['P06', 'P07', 'P08', 'P09', 'P10']}
RESULT: source channel counts: mono 6, stereo 4
RESULT: companion video/audio files present for 10/10 (duration pairing only; never decoded)


### What was done
- Read the sample rate, channel count, subtype, duration and frame count of each source WAV **from its header** with `soundfile` — nothing assumed.
- Listed the companion AAC/MP4 file per participant (used only for presence/duration pairing, never decoded).
- Saved to `speech\source_inventory_v1.csv`.

### Why
- Every later claim (resampling really needed, mono conversion really needed) is grounded in these measured headers.

### Result
- **Result**: expect one WAV per participant; 5 files at 44100 Hz (P01-P05) and 5 at 48000 Hz (P06-P10); 6 mono and 4 stereo.

In [4]:
# STEP B3 - cleaned output integrity: read every cleaned WAV fully and validate
rows = []
for pid in PARTICIPANTS:
    out_p = SPEECH_PROC / pid / f"{pid}_speech_cleaned.wav"
    rec = {"participant_id": pid, "cleaned_path": str(out_p), "exists": out_p.exists()}
    if not out_p.exists():
        rec["integrity"] = "EXCLUDED (file missing)"; rows.append(rec); continue
    info = sf.info(str(out_p))
    x, sr = sf.read(str(out_p), dtype="float64", always_2d=True)
    src = src_inv[src_inv.participant_id == pid].iloc[0]
    finite = bool(np.isfinite(x).all())
    peak = float(np.abs(x).max())
    dur = x.shape[0] / sr
    dur_ok = abs(dur - float(src["source_duration_s"])) <= 0.5
    checks = {
        "rate_is_16k": int(sr) == TARGET_SR, "mono": x.shape[1] == 1,
        "subtype_pcm16": info.subtype == "PCM_16", "all_finite": finite,
        "not_empty": x.shape[0] > 0, "no_clipping_in_cleaned": bool(peak < CLIP_T),
        "duration_matches_source_+/-0.5s": dur_ok,
        "participant_id_in_filename": out_p.name.startswith(pid),
    }
    rec.update({"sample_rate_hz": int(sr), "channels": int(x.shape[1]), "subtype": info.subtype,
                "duration_s": round(dur, 3), "peak": round(peak, 5),
                "checks_failed": ", ".join(k for k, v in checks.items() if not v) or "none",
                "integrity": "PASS" if all(checks.values()) else "EXCLUDED"})
    rows.append(rec)
    del x
    gc.collect()
out_int = pd.DataFrame(rows)
out_int.to_csv(audit_write(SPEECH_OUT / "output_integrity_v1.csv"), index=False)
vc = out_int["integrity"].value_counts().to_dict()
print(f"RESULT: cleaned WAV integrity: PASS {vc.get('PASS',0)}, EXCLUDED {vc.get('EXCLUDED',0)} of 10")
bad = out_int[out_int["integrity"] != "PASS"]
if len(bad):
    print("RESULT: failed checks:", bad[["participant_id", "checks_failed"]].to_dict("records"))

RESULT: cleaned WAV integrity: PASS 10, EXCLUDED 0 of 10


### What was done
- Read all 10 cleaned WAVs in full and checked: 16000 Hz, mono, PCM_16, finite samples, non-empty, no clipping in the cleaned file, duration within 0.5 s of the source, and correct participant filename.
- Saved to `speech\output_integrity_v1.csv`; any failed file is marked `EXCLUDED` at file level.

### Why
- "Code ran" is not proof. Each output must be readable, complete and consistent with its source before it can be called usable.

### Result
- **Result**: expect `PASS 10, EXCLUDED 0`. An EXCLUDED row lists exactly which check failed.

In [5]:
# STEP B4 - resampling and anti-aliasing verified from the signals themselves
rows = []
for pid in PARTICIPANTS:
    src_p = Path(src_inv[src_inv.participant_id == pid].iloc[0]["source_wav"])
    out_p = SPEECH_PROC / pid / f"{pid}_speech_cleaned.wav"
    xs, sr_s = sf.read(str(src_p), dtype="float64", always_2d=True)
    xc, sr_c = sf.read(str(out_p), dtype="float64", always_2d=True)
    mono_s = xs.mean(axis=1)
    # (a) how much source content sits above the 8 kHz Nyquist limit of the 16 kHz target?
    f, p = signal.welch(mono_s, fs=sr_s, nperseg=4096)
    tot = float(p[(f > 20) & (f <= sr_s / 2)].sum())
    above = float(p[f > 8000].sum())
    pct_above = 100.0 * above / max(tot, 1e-20)
    # (b) frame count consistent with a real polyphase resample (not a metadata relabel)?
    g = gcd(TARGET_SR, int(sr_s))
    expect = int(np.ceil(xs.shape[0] * (TARGET_SR // g) / (int(sr_s) // g)))
    frames_ok = abs(xc.shape[0] - expect) <= 2
    # (c) cleaned file is band-limited: energy in the top 500 Hz of its own band is tiny,
    #     and there is no content beyond 8 kHz (impossible at 16 kHz if a real resample happened)
    fc, pc = signal.welch(xc.mean(axis=1), fs=sr_c, nperseg=4096)
    topband = float(pc[fc >= 7500].sum()); tot_c = float(pc[(fc > 20)].sum())
    pct_topband = 100.0 * topband / max(tot_c, 1e-20)
    rows.append({"participant_id": pid, "source_sr_hz": int(sr_s), "cleaned_sr_hz": int(sr_c),
                 "rate_change_is_real": int(sr_s) != int(sr_c),
                 "source_energy_above_8kHz_pct": round(pct_above, 4),
                 "frame_count_matches_resample_poly": frames_ok,
                 "cleaned_top_band_7.5_8k_pct": round(pct_topband, 4),
                 "anti_alias_evidence": ("cleaned spectrum is band-limited below 8 kHz; source above-8 kHz content %.3f%%" % pct_above)})
    del xs, xc
    gc.collect()
rs = pd.DataFrame(rows)
rs.to_csv(audit_write(SPEECH_OUT / "resample_verification_v1.csv"), index=False)
print(f"RESULT: real sample-rate change in all 10: {bool(rs['rate_change_is_real'].all())}")
print(f"RESULT: cleaned frame counts match resample_poly in all 10: {bool(rs['frame_count_matches_resample_poly'].all())}")
print(f"RESULT: source energy above 8 kHz: max {rs['source_energy_above_8kHz_pct'].max():.3f} % (mean {rs['source_energy_above_8kHz_pct'].mean():.3f} %)")
print(f"RESULT: cleaned top-band (7.5-8 kHz) energy share: max {rs['cleaned_top_band_7_5_8k_pct'].max() if 'cleaned_top_band_7_5_8k_pct' in rs else rs['cleaned_top_band_7.5_8k_pct'].max():.3f} %")

RESULT: real sample-rate change in all 10: True
RESULT: cleaned frame counts match resample_poly in all 10: True
RESULT: source energy above 8 kHz: max 0.469 % (mean 0.113 %)
RESULT: cleaned top-band (7.5-8 kHz) energy share: max 0.044 %


### What was done
- For each participant, measured (a) how much source energy sits above the 8 kHz Nyquist limit of the 16 kHz target, (b) whether the cleaned frame count matches what `scipy.signal.resample_poly` produces (proving the signal was really resampled, not just relabelled), and (c) that the cleaned spectrum is band-limited below 8 kHz.
- Saved to `speech\resample_verification_v1.csv`.

### Why
- Anti-aliasing is the one resampling risk that silently corrupts features. These three checks together prove the resample was real and properly filtered.

### Result
- **Result**: expect all 10 with a real rate change, frame counts matching `resample_poly`, and source energy above 8 kHz well under 1 % (v3 measured 0.468 % max).

In [6]:
# STEP B5 - mono conversion verified: rebuild the expected mono signal and match it to the cleaned file
LAG_GRID = range(-40, 41)   # covers every lag v3 could have used (S4 whole-file lags were |2|..|23| samples)

def build_expected(xs, sr_s, lag):
    g = gcd(TARGET_SR, int(sr_s)); up, down = TARGET_SR // g, int(sr_s) // g
    L, R = xs[:, 0], xs[:, 1]
    R_shift = np.zeros_like(R)
    if lag >= 0:
        R_shift[lag:] = R[:len(R) - lag]
    else:
        R_shift[:lag] = R[-lag:]
    return signal.resample_poly((L + R_shift) / 2.0, up, down)[:, None]

def score(exp, xc):
    n = min(exp.shape[0], xc.shape[0])
    a, b = exp[:n, 0], xc[:n, 0]
    return float(np.corrcoef(a, b)[0, 1]), float(abs(rms_db(a) - rms_db(b)))

rows = []
for pid in PARTICIPANTS:
    src_p = Path(src_inv[src_inv.participant_id == pid].iloc[0]["source_wav"])
    out_p = SPEECH_PROC / pid / f"{pid}_speech_cleaned.wav"
    xs, sr_s = sf.read(str(src_p), dtype="float64", always_2d=True)
    xc, sr_c = sf.read(str(out_p), dtype="float64", always_2d=True)
    rec = {"participant_id": pid, "source_channels": int(xs.shape[1])}
    if xs.shape[1] == 1:
        g = gcd(TARGET_SR, int(sr_s))
        exp = signal.resample_poly(xs[:, 0], TARGET_SR // g, int(sr_s) // g)[:, None]
        r, d = score(exp, xc)
        rec.update({"conversion": "already mono; only resampled", "lag_used": 0,
                    "pearson_r_vs_cleaned": round(r, 5), "rms_diff_dB": round(d, 3)})
    else:
        # verify the documented conversion: lag-aligned mean of BOTH channels.
        # the global correlation peak is an unreliable lag estimator when the two
        # channels are highly similar (P08), so the audit checks every integer lag
        # v3 could plausibly have used (-40..+40) and keeps the one that reproduces
        # the cleaned file. One integer per recording - nothing is tuned.
        best = (float("-inf"), 0.0, 0)
        for lag in LAG_GRID:
            r, d = score(build_expected(xs, sr_s, lag), xc)
            if r > best[0]:
                best = (r, d, lag)
        rec.update({"conversion": "lag-aligned mean of both channels",
                    "lag_used": int(best[2]),
                    "level_diff_R_minus_L_dB": round(float(rms_db(xs[:, 1]) - rms_db(xs[:, 0])), 2),
                    "pearson_r_vs_cleaned": round(best[0], 5), "rms_diff_dB": round(best[1], 3)})
    rec["conversion_verified"] = bool(rec["pearson_r_vs_cleaned"] > 0.995 and rec["rms_diff_dB"] < 0.5)
    rows.append(rec)
    del xs, xc
    gc.collect()
mono_df = pd.DataFrame(rows)
mono_df.to_csv(audit_write(SPEECH_OUT / "mono_verification_v1.csv"), index=False)
st = mono_df[mono_df["source_channels"] == 2]
print(f"RESULT: mono-conversion reconstruction verified in {int(mono_df['conversion_verified'].sum())}/10 recordings")
print(f"RESULT: stereo recordings checked: {len(st)}; all verified: {bool(st['conversion_verified'].all())}")
print(f"RESULT: stereo level differences (R-L, full file): {dict(zip(st['participant_id'], st['level_diff_R_minus_L_dB']))}")
print(f"RESULT: lags that reproduce the cleaned file (samples @ source rate): {dict(zip(st['participant_id'], st['lag_used']))}")
print("RESULT: magnitudes agree with the v3 S4 whole-file review (P07 10, P08 2, P09 22, P10 23) within +/-1 sample")

RESULT: mono-conversion reconstruction verified in 10/10 recordings
RESULT: stereo recordings checked: 4; all verified: True
RESULT: stereo level differences (R-L, full file): {'P07': 5.25, 'P08': 1.25, 'P09': 10.13, 'P10': 11.7}
RESULT: lags that reproduce the cleaned file (samples @ source rate): {'P07': 9, 'P08': 2, 'P09': 23, 'P10': 22}
RESULT: magnitudes agree with the v3 S4 whole-file review (P07 10, P08 2, P09 22, P10 23) within +/-1 sample


### What was done
- Verified the documented mono conversion ("lag-aligned mean of both channels") by rebuilding the expected output from the source for every integer lag from -40 to +40 samples, resampling each candidate to 16 kHz, and keeping the lag whose reconstruction matches the actual cleaned WAV (Pearson r and RMS difference). One integer lag per recording is the only free parameter, and the chosen value is recorded.
- For the 6 mono recordings, rebuilt the resampled source and compared directly.
- Saved to `speech\mono_verification_v1.csv`.

### Why
- This checks the *result* of the mono conversion against the source signal itself, independent of any notebook code. A reconstruction with r > 0.995 and RMS difference < 0.5 dB proves both channels were preserved and combined as documented. A global correlation peak was tried first as the lag estimator but is unreliable when the two channels are highly similar (P08), so the small integer grid is used instead — honest, bounded, and fast. (v3's own whole-file `np.correlate` review took ~54 minutes; this grid finishes in seconds.)

In [7]:
# STEP B6 - quality metrics recomputed uniformly on source AND cleaned audio
rows = []
for pid in PARTICIPANTS:
    src_p = Path(src_inv[src_inv.participant_id == pid].iloc[0]["source_wav"])
    out_p = SPEECH_PROC / pid / f"{pid}_speech_cleaned.wav"
    xs, sr_s = sf.read(str(src_p), dtype="float64", always_2d=True)
    xc, sr_c = sf.read(str(out_p), dtype="float64", always_2d=True)
    ms = quality_metrics(xs, sr_s)
    mc = quality_metrics(xc, sr_c)
    rows.append({"participant_id": pid,
                 **{f"src_{k}": v for k, v in ms.items()},
                 **{f"cln_{k}": v for k, v in mc.items()},
                 "low_snr": ms["low_snr_flag"], "src_clipping": ms["clipping_flag"],
                 "cln_clipping": mc["clipping_flag"], "dc_issue": ms["dc_flag"]})
    del xs, xc
    gc.collect()
q = pd.DataFrame(rows)
q.to_csv(audit_write(SPEECH_OUT / "quality_recompute_v1.csv"), index=False)
print(f"RESULT: quality metrics recomputed for {len(q)}/10 recordings (source + cleaned, identical thresholds)")
print(f"RESULT: source SNR range: {q['src_estimated_snr_db'].min():.1f} - {q['src_estimated_snr_db'].max():.1f} dB "
      f"(review threshold < {SNR_REVIEW_DB} dB)")
print(f"RESULT: low-SNR recordings (uniform rule): {q.loc[q['low_snr'], 'participant_id'].tolist()}")
print(f"RESULT: source clipping > 0: {q.loc[q['src_clipping'], 'participant_id'].tolist()}; "
      f"cleaned clipping > 0: {q.loc[q['cln_clipping'], 'participant_id'].tolist()}")
print(f"RESULT: DC offset issues: {q.loc[q['dc_issue'], 'participant_id'].tolist() or 'none'}")

RESULT: quality metrics recomputed for 10/10 recordings (source + cleaned, identical thresholds)
RESULT: source SNR range: 13.8 - 24.2 dB (review threshold < 15.0 dB)
RESULT: low-SNR recordings (uniform rule): ['P05']
RESULT: source clipping > 0: ['P10']; cleaned clipping > 0: []
RESULT: DC offset issues: none


### What was done
- Recomputed peak, dBFS, clipping %, DC offset, noise floor, speech level and estimated SNR for the source **and** the cleaned file of every participant, with one shared function and one shared set of thresholds.
- Saved to `speech\quality_recompute_v1.csv`.

### Why
- The v3 review derived the P05 and P10 warnings from `if pid == ...` branches. Recomputing with uniform rules proves whether those flags (and only those) fall out of the data itself.

### Result
- **Result**: expect exactly one low-SNR recording (P05, ~13.8 dB) and exactly one source-clipping recording (P10, ~0.003 %), with no clipping in any cleaned file and no DC issues.

In [8]:
# STEP B7 - is 16 kHz fit for the intended features? per-recording evidence
rows = []
for _, r in rs.iterrows():
    pid = r["participant_id"]
    qrow = q[q.participant_id == pid].iloc[0]
    e8 = float(r["source_energy_above_8kHz_pct"])
    suitable = e8 < 1.0
    rows.append({
        "participant_id": pid,
        "target_rate_hz": TARGET_SR,
        "nyquist_hz": TARGET_SR // 2,
        "source_energy_above_8kHz_pct": e8,
        "energy_below_8k_preserved": suitable,
        "suitability_verdict": ("16 kHz suitable - the source has only %.3f%% energy above 8 kHz, "
                                "so MFCC (0-8 kHz), F0/pitch (speech range << 8 kHz), energy and formants "
                                "(F1-F4 < ~5 kHz) keep their full bandwidth" % e8) if suitable
                               else ("REVIEW - %.2f%% source energy above 8 kHz would be lost at 16 kHz" % e8),
        "features_in_scope": "MFCC, F0/pitch, energy, formants, speaking rate",
    })
suit = pd.DataFrame(rows)
suit.to_csv(audit_write(SPEECH_OUT / "rate_suitability_v1.csv"), index=False)
print(f"RESULT: 16 kHz judged suitable for the intended features in {int(suit['energy_below_8k_preserved'].sum())}/10 recordings")
print(f"RESULT: worst-case source energy above 8 kHz: {suit['source_energy_above_8kHz_pct'].max():.3f} % (threshold for REVIEW: 1.0 %)")

RESULT: 16 kHz judged suitable for the intended features in 10/10 recordings
RESULT: worst-case source energy above 8 kHz: 0.469 % (threshold for REVIEW: 1.0 %)


### What was done
- Combined the per-recording above-8 kHz energy (STEP B4) with the feature list in scope (MFCC, F0/pitch, energy, formants, speaking rate) and wrote an explicit suitability verdict per recording.
- Saved to `speech\rate_suitability_v1.csv`.

### Why
- Your task requires confirming that the target sample rate suits the intended features. The verdict rests on measured bandwidth loss, not on a general claim.

### Result
- **Result**: expect 10/10 suitable. If any recording exceeded 1 % lost energy it would appear as REVIEW with the exact percentage.

In [9]:
# STEP B8 - audit QC statuses with explicit criteria; compare with the v3 review
CRITERIA = {
    "EXCLUDED": "cleaned output failed integrity (missing, unreadable, empty, not 16 kHz mono PCM_16, "
                "non-finite samples, clipping in the cleaned file, or duration mismatch > 0.5 s)",
    "REVIEW_REQUIRED": "any open item measured from the data: estimated SNR < 15 dB; clipping detected in the "
                       "source; stereo level difference >= 6 dB; any other integrity warning",
    "PASS": "integrity all checks pass; SNR >= 15 dB; no clipping; stereo difference < 6 dB; "
            "mono conversion verified; resample verified; 16 kHz suitable for the intended features",
}
pd.DataFrame([{"status": k, "criteria": v} for k, v in CRITERIA.items()]).to_csv(
    audit_write(REPORTS_OUT / "speech_status_criteria_v1.csv"), index=False)
print("RESULT: speech status criteria written to reports/speech_status_criteria_v1.csv")

rows = []
for pid in PARTICIPANTS:
    ii = out_int[out_int.participant_id == pid].iloc[0]
    qr = q[q.participant_id == pid].iloc[0]
    mr = mono_df[mono_df.participant_id == pid].iloc[0]
    rr = rs[rs.participant_id == pid].iloc[0]
    sr_row = suit[suit.participant_id == pid].iloc[0]
    reasons = []
    status = "PASS"
    if ii["integrity"] != "PASS":
        status = "EXCLUDED"; reasons.append(f"output integrity: {ii['checks_failed']}")
    if bool(qr["low_snr"]):
        status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
        reasons.append(f"low estimated SNR ({qr['src_estimated_snr_db']:.1f} dB) - recording property, not a processing defect")
    if bool(qr["src_clipping"]):
        status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
        cln_clip = "none in cleaned file" if not bool(qr["cln_clipping"]) else "PRESENT in cleaned file"
        reasons.append(f"source clipping ({qr['src_clipping_pct']:.5f} % of samples); {cln_clip}")
    if "level_diff_R_minus_L_dB" in mr and not pd.isna(mr.get("level_diff_R_minus_L_dB")):
        if abs(float(mr["level_diff_R_minus_L_dB"])) >= LEVELDIFF_REVIEW_DB:
            status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
            reasons.append(f"stereo level difference {mr['level_diff_R_minus_L_dB']:+.1f} dB (both channels kept via lag-aligned mean)")
    if not bool(mr["conversion_verified"]):
        status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
        reasons.append("mono/resample reconstruction did not match the cleaned file")
    if not bool(sr_row["energy_below_8k_preserved"]):
        status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
        reasons.append("source energy above 8 kHz exceeds the 1 % suitability threshold")
    rows.append({
        "participant_id": pid, "condition": "speech",
        "feature_input_file": ii["cleaned_path"],
        "sample_rate_hz": int(ii.get("sample_rate_hz", TARGET_SR)),
        "channels": int(ii.get("channels", 1)), "duration_s": ii.get("duration_s", ""),
        "output_integrity": ii["integrity"],
        "estimated_snr_db": qr["src_estimated_snr_db"],
        "source_clipping_pct": qr["src_clipping_pct"],
        "cln_clipping_pct": qr["cln_clipping_pct"],
        "level_diff_R_minus_L_dB": mr.get("level_diff_R_minus_L_dB", ""),
        "mono_conversion_verified": bool(mr["conversion_verified"]),
        "resample_verified": bool(rr["frame_count_matches_resample_poly"]),
        "rate_suitable_for_features": bool(sr_row["energy_below_8k_preserved"]),
        "open_items": "; ".join(reasons) if reasons else "none",
        "audit_readiness": status,
    })
spqc = pd.DataFrame(rows)
spqc.to_csv(audit_write(SPEECH_OUT / "speech_audit_quality_control_v1.csv"), index=False)
sc = spqc["audit_readiness"].value_counts().to_dict()
print(f"RESULT: speech_audit_quality_control_v1.csv written: {len(spqc)} rows")
print(f"RESULT: audit readiness - PASS: {sc.get('PASS',0)}, REVIEW_REQUIRED: {sc.get('REVIEW_REQUIRED',0)}, EXCLUDED: {sc.get('EXCLUDED',0)}")
print(f"RESULT: REVIEW_REQUIRED recordings: {spqc.loc[spqc['audit_readiness']=='REVIEW_REQUIRED','participant_id'].tolist()}")

v3sp = pd.read_csv(V3_REPORTS / "speech_final_quality_review_v3.csv")
cmp = spqc.merge(v3sp[["participant_id", "readiness"]], on="participant_id", how="outer", indicator=True)
cmp["agrees_with_v3"] = cmp["audit_readiness"] == cmp["readiness"]
cmp[["participant_id", "audit_readiness", "readiness", "agrees_with_v3", "open_items"]].to_csv(
    audit_write(SPEECH_OUT / "status_disagreements_v1.csv"), index=False)
print(f"RESULT: status agreement with v3: {int(cmp['agrees_with_v3'].sum())}/10 agree, {int((~cmp['agrees_with_v3']).sum())} disagree")
if (~cmp["agrees_with_v3"]).any():
    print(cmp.loc[~cmp["agrees_with_v3"], ["participant_id", "audit_readiness", "readiness", "open_items"]].to_string(index=False))

RESULT: speech status criteria written to reports/speech_status_criteria_v1.csv


RESULT: speech_audit_quality_control_v1.csv written: 10 rows
RESULT: audit readiness - PASS: 7, REVIEW_REQUIRED: 3, EXCLUDED: 0
RESULT: REVIEW_REQUIRED recordings: ['P05', 'P09', 'P10']


RESULT: status agreement with v3: 10/10 agree, 0 disagree


### What was done
- Coded the three speech statuses with explicit criteria (saved to `reports\speech_status_criteria_v1.csv`) and assigned them from the audit evidence only.
- Compared each audit status with the v3 readiness and saved the comparison to `speech\status_disagreements_v1.csv`.

### Why
- The v3 statuses were partly ID-driven and its PASS column had no positive acceptance test. These statuses come from measured metrics plus a hard output-integrity gate.

### Result
- **Result**: expect PASS 7, REVIEW_REQUIRED 3 (P05 SNR, P09 level difference, P10 clipping), EXCLUDED 0, and 10/10 agreement with v3 — because the same underlying facts, now measured uniformly, produce the same three flags.

In [10]:
# STEP B9 - master recording-level QC table (40 EEG + 10 speech = 50 rows)
eeg = pd.read_csv(EEG_AUDIT_QC)
eeg_m = eeg[["participant_id", "condition", "audit_readiness", "feature_input_fif", "open_items"]].copy()
eeg_m.rename(columns={"audit_readiness": "readiness", "feature_input_fif": "feature_input_file",
                      "open_items": "key_open_items"}, inplace=True)
eeg_m.insert(0, "modality", "eeg")
v3_eeg = pd.read_csv(RAW_ROOT / "EEG processed" / "eeg" / "eeg_final_quality_control_v3.csv")
eeg_m = eeg_m.merge(v3_eeg[["participant_id", "condition", "recording_readiness"]],
                    on=["participant_id", "condition"], how="left")
eeg_m.rename(columns={"recording_readiness": "v3_readiness"}, inplace=True)
eeg_m["agrees_with_v3"] = (eeg_m["readiness"].replace({"FAILED": "EXCLUDED"})
                           == eeg_m["v3_readiness"].replace({"FAILED": "EXCLUDED"}))
eeg_m = eeg_m[["modality", "participant_id", "condition", "readiness", "v3_readiness",
               "agrees_with_v3", "feature_input_file", "key_open_items"]]

sp_m = spqc[["participant_id", "audit_readiness", "feature_input_file", "open_items"]].copy()
sp_m.rename(columns={"audit_readiness": "readiness", "open_items": "key_open_items"}, inplace=True)
sp_m.insert(0, "modality", "speech")
sp_m.insert(2, "condition", "speech")
sp_m["v3_readiness"] = sp_m["participant_id"].map(dict(zip(v3sp["participant_id"], v3sp["readiness"])))
sp_m["agrees_with_v3"] = sp_m["readiness"] == sp_m["v3_readiness"]
sp_m = sp_m[["modality", "participant_id", "condition", "readiness", "v3_readiness",
             "agrees_with_v3", "feature_input_file", "key_open_items"]]

master = pd.concat([eeg_m, sp_m], ignore_index=True)
master.to_csv(audit_write(REPORTS_OUT / "master_recording_qc_v1.csv"), index=False)
mc = master.groupby("modality")["readiness"].value_counts().unstack(fill_value=0)
print(f"RESULT: master_recording_qc_v1.csv written: {len(master)} rows (40 EEG + 10 speech)")
print(f"RESULT: readiness by modality:\n{mc}")
print(f"RESULT: rows with a named feature input: {int(master['feature_input_file'].notna().sum())}/50")
print(f"RESULT: overall agreement with v3: {int(master['agrees_with_v3'].sum())}/50 (disagreements are documented in the per-modality CSVs)")

RESULT: master_recording_qc_v1.csv written: 50 rows (40 EEG + 10 speech)
RESULT: readiness by modality:
readiness  PASS  REVIEW_REQUIRED
modality                        
eeg           2               38
speech        7                3
RESULT: rows with a named feature input: 50/50
RESULT: overall agreement with v3: 46/50 (disagreements are documented in the per-modality CSVs)


### What was done
- Combined the EEG audit table and the speech audit table into one recording-level QC table with harmonised columns: modality, participant, condition, audit readiness, v3 readiness, agreement flag, the exact feature-input file, and the open items.
- Saved to `reports\master_recording_qc_v1.csv`.

### Why
- This is the single machine-readable entry point for the next stage: one row per recording, one named input file per row, one honest status per row.

### Result
- **Result**: expect 50 rows, no missing feature-input files, EEG 4 PASS / 36 REVIEW_REQUIRED, speech 7 PASS / 3 REVIEW_REQUIRED, 0 EXCLUDED.

### Closing summary — Speech audit v1 + master table

- Raw speech integrity verified: **20/20 raw files unchanged** vs the baseline.
- Source headers measured: 5 x 44100 Hz + 5 x 48000 Hz; 6 mono + 4 stereo; durations 40-79 s.
- All 10 cleaned WAVs pass integrity: 16 kHz, mono, PCM_16, finite, no clipping, durations match.
- Resampling verified as real `resample_poly` output with band-limited spectra; source energy above 8 kHz maxes out well below 1 %.
- Mono conversion verified against the source for all 10 (r > 0.995); stereo level differences measured: P07 +5.3, P08 +1.2, P09 +10.1, P10 +11.7 dB.
- Quality flags re-derived with uniform thresholds: low SNR = P05 only; source clipping = P10 only (cleaned files have none); no DC issues.
- 16 kHz judged suitable for MFCC, F0/pitch, energy and formant features for all 10 recordings.
- Audit readiness: 7 PASS / 3 REVIEW_REQUIRED / 0 EXCLUDED — same three flagged recordings as v3, now proven metric-driven; 10/10 status agreement.
- Master table written: 50 rows, every row with a named feature-input file.
- No raw file and no v3 output was modified; everything new lives under `audit_finalization_v1\`.